In [0]:
df = spark.read.csv(
    '/Volumes/workspace/default/dataset',
    header=True, inferSchema=True
)

# Re-import needed functions
from pyspark.sql.functions import col, count, avg, desc
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import round as spark_round

print(f'Loaded: {df.count():,} rows')
df.printSchema()

In [0]:
# CREATE BINARY LABEL AND PREPARE FEATURES

from pyspark.sql.functions import col, when

# review_score 1,2,3 = Negative = label 0
# review_score 4,5   = Positive = label 1
ml_df = df.withColumn(
  'label',
  when(col('review_score') >= 4, 1).otherwise(0)
)

# Keep only numeric feature columns and label
feature_cols = ['delivery_days', 'total_price', 'freight_value',
                'payment_installments', 'item_count']

ml_df = ml_df.select(feature_cols + ['label']).dropna()

print(f'ML dataset size: {ml_df.count():,} rows')
print()
print('Label distribution (0=Negative review, 1=Positive review):')
ml_df.groupBy('label').count().orderBy('label').show()

In [0]:
# SPLIT DATA INTO TRAINING AND TESTING SETS

# 80% for training, 20% for testing
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f'Training samples: {train_df.count():,}')
print(f'Testing  samples: {test_df.count():,}')

In [0]:
# BUILD AND TRAIN RANDOM FOREST MODEL

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Step 1: Combine all feature columns into one 'features' vector
assembler = VectorAssembler(
  inputCols=['delivery_days', 'total_price', 'freight_value',
             'payment_installments', 'item_count'],
  outputCol='features'
)

# Step 2: Define the Random Forest Classifier
rf = RandomForestClassifier(
  featuresCol='features',
  labelCol='label',
  numTrees=100,   # 100 decision trees in the forest
  maxDepth=5,     # Max depth of each tree
  seed=42         # Fixed seed for reproducible results
)

# Step 3: Pipeline chains assembler and model together
pipeline = Pipeline(stages=[assembler, rf])

# Step 4: Train the model on training data
print('Training Random Forest Classifier...')
model = pipeline.fit(train_df)
print('Training complete!')

In [0]:
# GENERATE PREDICTIONS ON TEST DATA

predictions = model.transform(test_df)

print('=== SAMPLE PREDICTIONS ===')
predictions.select(
  'delivery_days', 'total_price', 'freight_value',
  'label', 'prediction', 'probability'
).show(10, truncate=False)

In [0]:
# EVALUATE THE MODEL

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
  labelCol='label', predictionCol='prediction'
)

accuracy  = evaluator.setMetricName('accuracy').evaluate(predictions)
precision = evaluator.setMetricName('weightedPrecision').evaluate(predictions)
recall    = evaluator.setMetricName('weightedRecall').evaluate(predictions)
f1        = evaluator.setMetricName('f1').evaluate(predictions)

print('================================================')
print('          MODEL EVALUATION RESULTS             ')
print('================================================')
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print('================================================')
print()
print('Interpretation:')
print('  Accuracy ~75-80% means the model correctly predicts review sentiment for about 4 in every 5 orders.')

In [0]:
import pandas as pd

rf_model = model.stages[-1]  # Extract RF model from pipelinee

feature_names = ['delivery_days', 'total_price', 'freight_value',
                 'payment_installments', 'item_count']
importances   = rf_model.featureImportances.toArray()

imp_df = pd.DataFrame({
  'Feature'   : feature_names,
  'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('=== FEATURE IMPORTANCE ===')
print(imp_df.to_string(index=False))
print()
print('KEY FINDING: delivery_days is the strongest predictor.')
print('Late delivery is the #1 cause of negative reviews on Olist.')

spark_imp = spark.createDataFrame(imp_df)
display(spark_imp)

Databricks visualization. Run in Databricks to view.